# KITTI evaluation + val-sample export (Kaggle, CPU is enough)
No GPU needed: set Accelerator to None, Internet On, attach the KITTI dataset. Does not retrain anything.
1. Recreates the same 80/20 split as training (seed 42).
2. Runs distance evaluation Setting A (oracle boxes) on all val frames.
3. Zips 200 random val frames to download to the Mac.

In [ ]:
# 1. Settings
KITTI = "/kaggle/input/datasets/garymk/kitti-3d-object-detection-dataset/training"
N_EXPORT = 200

In [ ]:
# Check the dataset path first. Should list: calib image_2 label_2 velodyne (and 7481 labels)
!ls {KITTI}
!ls {KITTI}/label_2 | wc -l
# If this says 'No such file', find the real path with:  !find /kaggle/input -maxdepth 4 -type d

In [ ]:
# 2. Get the code and dependencies
import os
if not os.path.exists("/kaggle/working/kitti-camera-lidar-fusion"):
    !git clone https://github.com/seanpan-1321/kitti-camera-lidar-fusion.git /kaggle/working/kitti-camera-lidar-fusion
%cd /kaggle/working/kitti-camera-lidar-fusion
!git pull
!pip install -q tabulate

In [ ]:
# 3. Same split as training (deterministic: seed 42), only writes two small text files
!python scripts/make_split.py --kitti_root {KITTI} --out /kaggle/working/splits
!wc -l /kaggle/working/splits/val.txt

In [ ]:
# 4. Setting A (oracle boxes) on ALL val frames. Paste the printed table back to Claude.
!python scripts/eval_distance.py --kitti_root {KITTI} --split_file /kaggle/working/splits/val.txt --out /kaggle/working/metrics --tag A_oracle_fullval

In [ ]:
# 5. Zip 200 random val frames to download (about 600 MB). Download it from the Output tab.
!python scripts/export_sample.py --kitti_root {KITTI} --split_file /kaggle/working/splits/val.txt --n {N_EXPORT} --out /kaggle/working/kitti_val_sample.zip